# Sesión 5 · Auditable: un registro que otro pueda verificar

**Objetivo.** Que cada número del informe salga de una ejecución, que otro agente pueda reproducirlo,
y evaluar el agente final con datos nuevos. **Tiempo:** 80 minutos.

In [ ]:
# --- Configuración: correr una vez al empezar ---
import os, sys, json
from pathlib import Path
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))
import pandas as pd
import matplotlib.pyplot as plt
from agente.llm import desde_entorno
from agente.harness import ejecutar_agente, correr_varias, leer_traza, con_especificacion, SISTEMA_BASE
from agente.evaluacion import tarjeta, huerfanos, guardar, cargar, cargar_respuesta, diagnosticos_ejecutados

# os.environ["LLM_PROVEEDOR"] = "ollama"   # o "anthropic" / "openai"; si no lo configurás, se usa el simulador
# os.environ["LLM_MODELO"] = "qwen3.5:0.8b"   # el modelo que descargaste (ver INSTALACION.md)
llm = desde_entorno()
df = pd.read_csv(RAIZ / "datos" / "programa_x.csv")
TRAZAS = RAIZ / "trazas" / "s5"
RES = RAIZ / "resultados"; RES.mkdir(exist_ok=True)
VERDAD, TOLERANCIA = cargar_respuesta(RAIZ)
PREGUNTA = "¿Cuánto aumentó el programa el ingreso posterior de quienes participaron?"
ESPEC = (RAIZ / "especificaciones" / "especificacion.md").read_text(encoding="utf-8")
SISTEMA = con_especificacion(ESPEC)
from agente import herramientas as h
from agente.revisor import revisar

## 1. ¿De dónde sale cada número? (15 min)

Estas afirmaciones de la sesión 4 tienen números que no aparecen en ninguna salida ejecutada.
O el agente los calculó «de cabeza», o los inventó.

In [ ]:
prev = cargar(RES / "s4_a.json")
huerfanos(prev)

In [ ]:
TRAZABILIDAD = """
TRAZABILIDAD: cada número que reportes tiene que ser trazable a una salida que imprimiste.
No reportes p-valores ni porcentajes que no hayas calculado e impreso.
"""
SISTEMA5 = SISTEMA + "\n" + h.INSTRUCCIONES + TRAZABILIDAD
res = correr_varias(5, PREGUNTA, df, llm, sistema=SISTEMA5, herramientas={"h": h},
                    carpeta_trazas=TRAZAS, etiqueta="s5")
guardar(res, RES / "s5.json")
huerfanos(res)

## 2. Lo que dice contra lo que hizo (10 min)

In [ ]:
pd.DataFrame({"corrida": res.corrida,
              "declarados": res.diagnosticos.apply(sorted),
              "ejecutados": res.traza.apply(lambda t: sorted(diagnosticos_ejecutados(t)))})

## 3. Un revisor independiente (20 min)

Otro agente recibe solo los datos, la especificación y el informe (nunca la traza) y trata de reproducirlo.

In [ ]:
for _, fila in res.head(2).iterrows():
    informe = json.loads(fila[["estimacion", "ic_inferior", "ic_superior", "metodo",
                               "diagnosticos", "afirmaciones", "abstiene"]].to_json())
    rev = revisar(informe, ESPEC, df, llm, herramientas={"h": h}, carpeta_trazas=TRAZAS)
    print(f"--- {fila.corrida} ---")
    print("\n".join(rev.respuesta["afirmaciones"]) if rev.respuesta else rev.error)

## 4. Evaluación final con datos nuevos (25 min)

El docente entrega `programa_y.csv` y `mapa_columnas.json`: otra muestra, otros nombres de columnas,
otra moneda. Copialos a `evaluacion_final/` y corré tu agente **sin tocar la especificación**.

In [ ]:
CARPETA = RAIZ / "evaluacion_final"
mapa = json.loads((CARPETA / "mapa_columnas.json").read_text(encoding="utf-8"))
dfy = pd.read_csv(CARPETA / "programa_y.csv").rename(columns=mapa)
res_y = correr_varias(10, PREGUNTA, dfy, llm, sistema=SISTEMA5, herramientas={"h": h},
                      carpeta_trazas=TRAZAS, etiqueta="s5-final")
guardar(res_y, RES / "s5_final.json")
res_y[["metodo", "estimacion", "ic_inferior", "ic_superior", "abstiene"]]

Cuando todos los equipos guardaron sus corridas, el docente entrega `respuesta_holdout.json`.

In [ ]:
r = json.loads((CARPETA / "respuesta_holdout.json").read_text(encoding="utf-8"))
inicial = tarjeta(cargar(RES / "s1.json"), VERDAD, TOLERANCIA)
final = tarjeta(res_y, r["efecto_verdadero"], r["tolerancia"])
inicial.merge(final, on=["pilar", "métrica"], how="outer", suffixes=(" · sesión 1", " · final"))

## 5. El memo (para entregar)

Dos páginas, con la tarjeta de la sesión 1 y la final como anexo. Respondé:

1. ¿Qué decisiones dejarías que este agente tome sin revisión humana? ¿Cuáles no?
2. ¿Qué pilar mejoró más? ¿Cuál menos, y por qué?
3. Si tuvieras que usarlo mañana con datos de tu trabajo, ¿qué agregarías primero?